<div align="center">
  <a href="https://colab.research.google.com/github/PrunaAI/ai-efficiency-courses/blob/main/solutions/06-use_data_llm_quantization.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</div>

---
**💡 Tip**: Click the button above to open this notebook in Google Colab for free GPU access!

## Installation

This notebook includes automatic setup cells that will install the project from git repository with UV.

**Note**: Run the setup cells below before starting the exercises.

In [ ]:
# Install project directly from git repository
!uv pip install git+https://github.com/PrunaAI/ai-efficiency-courses.git

## Utility cells

During the course, we'll leverage some course utilities to streamline our workflow. These utilities are located in the `course` package, which can simply be imported given that we installed the project from git repository above.
You can find the source code [here](https://github.com/PrunaAI/ai-efficiency-courses/tree/main/course).

These utilities will help us:
- Load and manage lists of model ids that we have verified to work.
- Generate informative plots for model analysis.
- Iterate efficiently over evaluation and model configuration options.

Let's first load our models. We will use `SMALL_MODEL_IDS`, which are sub 1B parameters which should be easy to download and load into memory. We recommend starting with these smaller models but feel free to experiment with other models until you reach your GPU memory limit!

In [ ]:
from course import SMALL_MODEL_IDS, MEDIUM_MODEL_IDS, LARGE_MODEL_IDS, ALL_MODEL_IDS

MODEL_IDS = SMALL_MODEL_IDS
# MODEL_IDS = MEDIUM_MODEL_IDS
# MODEL_IDS = LARGE_MODEL_IDS
# MODEL_IDS = ALL_MODEL_IDS

MODEL_IDS

We also recommend to set a custom cache directory for models. Loading models can take significant disk space. To avoid filling up your default disk, we recommend setting a custom cache directory for downloaded models. You can do this by running the following in a terminal or in a notebook cell:

In [ ]:
# Replace <path_to_cache> with your desired cache path
import os

CACHE_PATH = "<path_to_cache>"
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH

You can also clear the cache by running the following cell:

In [ ]:
from course.models import clear_cache

clear_cache(CACHE_PATH)

# 06: Use Data in LLM Quantization

Welcome to this new unit of the AI Efficiency course! 🚀

In this tutorial, we will explore data can be used during the quantization process. When data is available, it is reasonable to make use of it to improve the quantized models. The content from the chapter 4 [slides](slides/04-quantize_language_models.pdf) will help you to go through this notebook.

By the end of this unit, you will:
- Understand how to use data to quantize LLMs.
- Evaluate the impact in the quantization of:
    - no data,
    - in-distribution data,
    - out-of-distribution data,
    - synthetic data

Let's get started on incorporating data during LLM quantization!

## 1. Imports

As we've already installed the project, we can import the necessary libraries. We will be using `torch` and `transformers` for this tutorial as interfaces to the model and tokenizer. On top of that, we will be using `matplotlib` for basic plotting. We recommend to checkout the [Pruna documentation](https://docs.pruna.ai/en/stable/docs_pruna/user_manual/evaluate.html) for access to AI efficiency functions.

In [9]:
import copy
import random

from datasets import load_dataset
from pruna import SmashConfig, smash
from pruna.data.utils import split_train_into_train_val_test
from pruna.evaluation.metrics.metric_torch import TorchMetricWrapper
from transformers import AutoModelForCausalLM, AutoTokenizer

Beyond external libraries, this course comes with the `course` local package which contains a lot of utils that you can use in the notebooks. We will be using `evaluate_model` to evaluate the model.

In [6]:
from course import evaluate_model, create_single_plot

## 2. Use Data in LLM Quantization

### 2.1 Evaluate the Base Model Quality

**Why is this important?**
 Understanding baseline model performance helps establish a reference point for comparing
 quantized versions. Evaluating on different datasets and models provides insight into
 how quantization impacts vary across contexts.

**Your tasks:**
 - Evaluate the base model quality with the perplexity metric on the WikiText dataset
 - Repeat the experiment with other LLMs and/or datasets

**What to think about:**
 - Check that the number are comparable to what you found in other notebooks, paper, repos.

In [10]:
def smash_evaluate_perplexity(
    model_id, tokenizer=None, smash_config=None, dataset="WikiText"
):
    """
    Compress using quantizationand evaluate the perplexity of the model.

    Args:
        model_id: The model to evaluate.
        tokenizer: The tokenizer to use.
        smash_config: The configuration to use for the model.
        dataset: The dataset to use for the evaluation.

    Returns:
        A dictionary with the perplexity of the model.
    """
    ### To Complete ###
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto").cuda()
    model_copy = copy.deepcopy(model)

    if tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(model_id)

    if smash_config:
        model_copy = smash(model_copy, smash_config)
    else:
        model_copy = model_id

    results = evaluate_model(
        model_id_or_pruna_model=model_copy,
        tokenizer_id_or_tokenizer=tokenizer,
        metrics=[TorchMetricWrapper(metric_name="perplexity", call_type="single")],
        dataset=dataset,
    )
    ### End of To Complete ###
    return results

In [11]:
# Select the model to evaluate
model_id = MODEL_IDS[0]
### To Complete ###
baseline_results =  smash_evaluate_perplexity(model_id=model_id)
### End of To Complete ###
print(baseline_results)

INFO - Using best available device: 'cuda'
INFO - Using call_type: y_gt for metric perplexity
INFO - Using best available device: 'cuda'
INFO - Using max_seq_len of tokenizer: None
INFO - Testing compatibility with text_generation_collate...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO - Using provided list of metric instances.
INFO - Using best available device: 'cuda'
INFO - Evaluating a base model.
INFO - Detected transformers model. Using TransformerHandler.
- The first element of the batch is passed as input.
- The generated outputs are expected to have .logits attribute.
INFO - Evaluating stateful metrics.
INFO - Evaluating isolated inference metrics.


[MetricResult(name='perplexity', params={'_defaults': {}, 'metric': Perplexity(), 'update_fn': <function default_update at 0x700757b6a440>, 'call_type': 'y_gt', 'metric_name': 'perplexity', 'higher_is_better': False}, result=49.46204376220703)]



### 2.2 Quantize without Data

**Why is this important?**
Understanding how quantization without data affects model performance helps establish a baseline
for comparing more sophisticated quantization approaches. This provides insight into the value
of data-driven quantization methods.

**Your tasks:**
- Quantize the LLM with GPTQ, which performs naive linear quantization without using any data
- Evaluate the quantized model quality using perplexity on WikiText dataset
- Compare results to the base model performance

**What to think about:**
- How much does performance degrade with naive quantization?
- What are the tradeoffs between model size reduction and quality loss?
- In what scenarios might data-free quantization be preferable?

In [12]:
### To Complete ###
smash_config = SmashConfig({
    "gptq" : {
        "gptq_weight_bits": "4"
    }
})
smash_config.add_tokenizer(model_id)

results_without_data = smash_evaluate_perplexity(
    model_id=model_id, smash_config=smash_config
)
### End of To Complete ###
print(results_without_data)

/tmp/ipykernel_15225/1052334567.py:2: DeprecationWarning: max_batch_size is deprecated. Please use batch_size instead.
  smash_config = SmashConfig({
INFO - Using best available device: 'cuda'
INFO - Using best available device: 'cuda'
INFO - Using call_type: y_gt for metric perplexity
INFO - Testing compatibility with text_generation_collate...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO - Using provided list of metric instances.
INFO - Using best available device: 'cuda'
INFO - Evaluating a smashed model.
INFO - Detected transformers model. Using TransformerHandler.
- The first element of the batch is passed as input.
- The generated outputs are expected to have .logits attribute.
INFO - Evaluating stateful metrics.
INFO - Evaluating isolated inference metrics.


[MetricResult(name='perplexity', params={'_defaults': {}, 'metric': Perplexity(), 'update_fn': <function default_update at 0x700757b6a440>, 'call_type': 'y_gt', 'metric_name': 'perplexity', 'higher_is_better': False}, result=49.46204376220703)]



### 2.3 Quantize with In-Distribution Data

**Why is this important?**
Understanding how data-driven quantization affects model performance helps evaluate the benefits
of using in-distribution data during quantization. This provides insights into optimizing the
quantization process for better model quality.

**Your tasks:**
- Quantize the LLM with GPTQ using in-distribution data from WikiText
- Evaluate the quantized model quality using perplexity on WikiText dataset
- Compare results to data-free quantization performance

**What to think about:**
- How much does in-distribution data improve quantization quality?
- What are the tradeoffs between data collection effort and quality gains?
- In what scenarios is data-driven quantization worth the additional complexity?

In [13]:
### To Complete ###
smash_config = SmashConfig({
    "gptq" : {
        "gptq_weight_bits": "4"
    }
})
smash_config.add_tokenizer(MODEL_IDS[0])
smash_config.add_data("WikiText", tokenizer=model_id)

results_in_distribution = smash_evaluate_perplexity(
    model_id=model_id, smash_config=smash_config
)
### End of To Complete ###
print(results_in_distribution)

/tmp/ipykernel_15225/28090444.py:2: DeprecationWarning: max_batch_size is deprecated. Please use batch_size instead.
  smash_config = SmashConfig({
INFO - Using best available device: 'cuda'
INFO - Testing compatibility with text_generation_collate...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO - Using best available device: 'cuda'
INFO - Using call_type: y_gt for metric perplexity
INFO - Testing compatibility with text_generation_collate...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO - Using provided list of metric instances.
INFO - Using best available device: 'cuda'
INFO - Evaluating a smashed model.
INFO - Detected transformers model. Using TransformerHandler.
- The first element of the batch is passed as input.
- The generated outputs are expected to have .logits attribute.
INFO -

[MetricResult(name='perplexity', params={'_defaults': {}, 'metric': Perplexity(), 'update_fn': <function default_update at 0x700757b6a440>, 'call_type': 'y_gt', 'metric_name': 'perplexity', 'higher_is_better': False}, result=49.46204376220703)]



### 2.4 Quantize with Varying Amounts of In-Distribution Data

**Why is this important?**
Understanding how the amount of in-distribution data (i.e. data similar to the data met during inference) affects quantization helps optimize the data collection process.
This provides insights into the minimum data requirements needed for effective quantization.

**Your tasks:**
- Quantize the LLM with GPTQ or AWQ using a small/large subset of WikiText data
- Evaluate the quantized model quality using perplexity on WikiText dataset
- Compare results to quantization with full dataset

**What to think about:**
- How much in-distribution data is needed for good quantization results?
- What is the relationship between data amount and model quality?
- At what point do additional data samples provide diminishing returns?

In [14]:
train_ds, val_ds, test_ds = load_dataset(
    "mikasenghaas/wikitext-2", split=["train", "validation", "test"]
)
train_ds = train_ds.select(range(1000))
### To Complete ###
val_ds = val_ds.select(range(100))
test_ds = test_ds.select(range(100))

smash_config = SmashConfig({
    "gptq" : {
        "gptq_weight_bits": "4"
    }
})
smash_config.add_tokenizer(model_id)
smash_config.add_data((train_ds, val_ds, test_ds), collate_fn="text_generation_collate")

results_varying_data = smash_evaluate_perplexity(
    model_id=model_id, smash_config=smash_config
)
### End of To Complete ###
print(results_varying_data)

/tmp/ipykernel_15225/3708593542.py:9: DeprecationWarning: max_batch_size is deprecated. Please use batch_size instead.
  smash_config = SmashConfig({
INFO - Using best available device: 'cuda'
INFO - Using max_seq_len of tokenizer: None
INFO - Testing compatibility with text_generation_collate...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO - Using best available device: 'cuda'
INFO - Using call_type: y_gt for metric perplexity
INFO - Testing compatibility with text_generation_collate...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO - Using provided list of metric instances.
INFO - Using best available device: 'cuda'
INFO - Evaluating a smashed model.
INFO - Detected transformers model. Using TransformerHandler.
- The first element of the batch is passed as input.
- The generated outputs 

[MetricResult(name='perplexity', params={'_defaults': {}, 'metric': Perplexity(), 'update_fn': <function default_update at 0x700757b6a440>, 'call_type': 'y_gt', 'metric_name': 'perplexity', 'higher_is_better': False}, result=49.46204376220703)]



### 2.5 Quantize LLM with Random Data

**Why is this important?**
Understanding how random data affects quantization helps determine if data quality matters.
This provides insights into whether collecting high-quality in-distribution data is necessary.

**Your tasks:**
- Quantize the LLM with GPTQ or AWQ using randomly generated text data
- Evaluate the quantized model quality using perplexity on WikiText dataset
- Compare results to quantization with no data or real data

**What to think about:**
- Does random data provide effective quantization?
- How does model quality compare to using no data or real text data?
- What are the implications for data collection requirements?

In [15]:
train_ds = [
    {"text": "".join([chr(random.randint(97, 122)) for _ in range(100)])}
    for _ in range(1000)
]
### To Complete ###
val_ds = [
    {"text": "".join([chr(random.randint(97, 122)) for _ in range(100)])}
    for _ in range(100)
]
test_ds = [
    {"text": "".join([chr(random.randint(97, 122)) for _ in range(100)])}
    for _ in range(100)
]

smash_config = SmashConfig({
    "gptq" : {
        "gptq_weight_bits": "4"
    }
})
smash_config.add_tokenizer(model_id)
smash_config.add_data((train_ds, val_ds, test_ds), collate_fn="text_generation_collate")

results_random_data = smash_evaluate_perplexity(
    model_id=model_id, smash_config=smash_config
)
### End of To Complete ###
print(results_random_data)

/tmp/ipykernel_15225/1968094613.py:15: DeprecationWarning: max_batch_size is deprecated. Please use batch_size instead.
  smash_config = SmashConfig({
INFO - Using best available device: 'cuda'
INFO - Testing compatibility with text_generation_collate...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO - Using best available device: 'cuda'
INFO - Using call_type: y_gt for metric perplexity
INFO - Testing compatibility with text_generation_collate...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO - Using provided list of metric instances.
INFO - Using best available device: 'cuda'
INFO - Evaluating a smashed model.
INFO - Detected transformers model. Using TransformerHandler.
- The first element of the batch is passed as input.
- The generated outputs are expected to have .logits attribute.
INF

[MetricResult(name='perplexity', params={'_defaults': {}, 'metric': Perplexity(), 'update_fn': <function default_update at 0x700757b6a440>, 'call_type': 'y_gt', 'metric_name': 'perplexity', 'higher_is_better': False}, result=49.46204376220703)]



### 2.6 Quantize LLM with Out-Of-Distribution Data

**Why is this important?**
Understanding how out-of-distribution data (i.e. data which is different from the data met during inference) affects quantization helps optimize data selection.
This provides insights into whether domain-specific data is needed for effective quantization.

**Your tasks:**
- Quantize the LLM with GPTQ or AWQ using BookCorpus dataset
- Evaluate the quantized model quality using perplexity on WikiText dataset
- Compare results to quantization with in-distribution data

**What to think about:**
- How does out-of-distribution data affect quantization quality?
- What is the relationship between data domain and model quality?
- Is domain-specific data necessary for good quantization results?

In [16]:
train_ds = load_dataset("SamuelYang/bookcorpus")["train"]
### To Complete ###
train_ds, val_ds, test_ds = split_train_into_train_val_test(train_ds, seed=42)
train_ds = train_ds.select(range(1000))
val_ds = val_ds.select(range(100))
test_ds = test_ds.select(range(100))

smash_config = SmashConfig({
    "gptq" : {
        "gptq_weight_bits": "4"
    }
})
smash_config.add_tokenizer(model_id)
smash_config.add_data((train_ds, val_ds, test_ds), collate_fn="text_generation_collate")

results_out_of_distribution = smash_evaluate_perplexity(
    model_id=model_id, smash_config=smash_config
)
### End of To Complete ###
print(results_out_of_distribution)

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

/root/sdiazlor/ai-efficiency-courses/.venv/lib/python3.10/site-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)
INFO - Loaded only training, splitting train 80/10/10 into train, validation and test...
/tmp/ipykernel_15225/4185575345.py:8: DeprecationWarning: max_batch_size is deprecated. Please use batch_size instead.
  smash_config = SmashConfig({
INFO - Using best available device: 'cuda'
INFO - Testing compatibility with text_generation_collate...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO - Using best available device: 'cuda'
INFO - Using call_type: y_gt for metric perplexity
INFO - Testing compatibility with text_generation_collate...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO -

[MetricResult(name='perplexity', params={'_defaults': {}, 'metric': Perplexity(), 'update_fn': <function default_update at 0x700757b6a440>, 'call_type': 'y_gt', 'metric_name': 'perplexity', 'higher_is_better': False}, result=49.46204376220703)]



### 2.6 Compare the Results

**Your tasks:**
- Create a plot, using `create_single_plot`, of the results of the different experiments.

**What to think about:**
- Which are the differences?
- How data influences the results?

In [17]:
results = {
    "Baseline": baseline_results,
    "Without Data": results_without_data,
    "In Distribution": results_in_distribution,
    "Varying Data": results_varying_data,
    "Random Data": results_random_data,
    "Out of Distribution": results_out_of_distribution,
}
results_dict = {name: res[0].result for name, res in results.items()}

### To Complete ###
create_single_plot(
    data_dict=results_dict,
    x_label="Metric",
    y_label="Value",
    title="Model Evaluation Results"
)
### End of To Complete ###

## Conclusion: What We've Learned About LLM on CPU and GPU

In this module, we explored quantizing LLMs on without data or with different types of data. Here are the key findings:

- **Real Data > Synthetic Data > No Data:**
  Using real data during quantization improved the performance of the quantized models. No data can still performs fairly good.

- **In-distribution Data > Out-of-distribution Data:**
  Using data close to data that will be used during inference to quantize a model improves performance of the quantized model.

- **More Data Helps a Bit:**
  While more data achieves better results, it is not strictly required to have huge datasets to achieve very good results

### Next Steps: Leverage Data During Fine-Tuning

Now that you understand how data affects quantization quality, you can make informed decisions about what data to use when quantizing models. The next sections will explore additional techniques to leverage data during fine-tuning (which can serve as recovery after quantization).

👉 **Continue to the next notebook:**

[07-finetune_llm.ipynb on GitHub](https://github.com/PrunaAI/ai-efficiency-courses/blob/main/exercises/07-finetune_llm.ipynb).